In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression

In this example, ﻿Z﻿ = the likelihood of a biking accident, ﻿Y﻿ = speed, and ﻿X﻿= trail difficulty. We assume that ﻿X﻿decreases ﻿Y﻿ causally because people decrease their speed on difficult trails. In addition, ﻿Y﻿ and ﻿X﻿ both increase ﻿Z﻿ causally because fast biking on difficult trails leads to accidents. Difficulty will be on a scale from 0 to 1, speed in miles per hour, and likelihood of an accident also on a scale from 0 to 1. (Based on the numbers, I'd say these trails are quite challenging!) 

In [2]:
# Code to create data for this question 
num = 100000 
 
difficulty = np.random.uniform(0, 1, (num,)) 
 
speed = np.maximum(np.random.normal(15, 5, (num, )) - difficulty * 10, 0) 
 
accident = np.minimum(np.maximum(0.03 * speed + 0.4 * difficulty + np.random.normal(0, 0.3, (num,)), 0), 1) 
 
df = pd.DataFrame({'difficulty': difficulty, 'speed': speed, 'accident': accident}) 

Question 1


Use ﻿X﻿ to predict ﻿Y﻿many times via regression with different data sets. Use many samples in each prediction. Which is closest to the average coefficient of ﻿X﻿ if you do the experiment enough times? 

In [3]:
def make_data(num=100000):
    difficulty = np.random.uniform(0, 1, (num,))
    speed = np.maximum(np.random.normal(15, 5, (num,)) - difficulty * 10, 0)
    accident = np.minimum(np.maximum(0.03 * speed + 0.4 * difficulty
                                     + np.random.normal(0, 0.3, (num,)), 0), 1)
    return pd.DataFrame({'difficulty': difficulty, 'speed': speed, 'accident': accident})

coefs_x = []
for _ in range(50):
    d = make_data()
    model = LinearRegression().fit(d[['difficulty']], d['speed'])
    coefs_x.append(model.coef_[0])

print("Avg coefficient of X (Y ~ X):", np.mean(coefs_x))

Avg coefficient of X (Y ~ X): -9.670807024110058



## The setup (causal graph)

```
X (difficulty) ──(−)──► Y (speed)
      │                    │
      └──(+)──► Z ◄──(+)───┘
              (accident)
```

- In the code, the true effect of difficulty on speed is **−10**: `speed = N(15,5) − 10·difficulty`.
- `Z` is a **collider**, since both `X` and `Y` cause it.

## Question 1: regress Y on X, repeated many times

"Different data sets" means you regenerate the data inside a loop, fit each time, and average the coefficients.


**Result: ≈ −9.65**

It's close to the true −10, but a bit smaller in magnitude. The cause is `np.maximum(..., 0)`: speed can't go below zero. On hard trails, some speeds would have been negative and get clipped up to 0, which flattens the slope a little. If your options include both −10 and about −9.6, pick the one closest to −9.65.


Question 2
 

Then use ﻿X﻿ and ﻿Z﻿ to predict﻿Y﻿ many times via regression with different datasets. Which of these is closest to the average coefficient of ﻿X﻿? 



Note: In practice, should we run such a regression? We are controlling for ﻿Z﻿, but ﻿Z﻿is a collider. That is, ﻿Y﻿and ﻿X﻿ both cause ﻿Z﻿. Should we control of it or are we better off ignoring ﻿Z﻿? Why or why not? 

In [4]:
coefs_xz = []
for _ in range(50):
    d = make_data()
    model = LinearRegression().fit(d[['difficulty', 'accident']], d['speed'])
    coefs_xz.append(model.coef_[0])   # coef_[0] is difficulty

print("Avg coefficient of X (Y ~ X + Z):", np.mean(coefs_xz))

Avg coefficient of X (Y ~ X + Z): -10.330059080329715


**Result: ≈ −10.32**

Make sure you read `coef_[0]` (difficulty) and not `coef_[1]` (accident).

## Should we control for Z? (the conceptual part)

**No.** Ignore `Z` and use the Question 1 model.

**Why controlling for a collider biases the estimate:**
- Before conditioning, the path X → Z ← Y is **blocked**, because a collider blocks a path by default.
- Adding `Z` to the regression **opens** that path. That creates a spurious association between `X` and `Y`.
- **Example:** look only at rides with the same accident likelihood. If a trail is hard (high `X`), that likelihood is already "explained" by the difficulty, so the rider must have been going slower (lower `Y`). Holding `Z` fixed makes `X` look more negative for speed than it really is.
- That's why the coefficient moves from −9.65 to −10.32. It's **further from the truth** for this data, not closer. It only lands nearer the theoretical −10 by coincidence.

**Rule to remember:** control for **confounders** (common causes of `X` and `Y`). Do **not** control for **colliders** (common effects) or mediators. Here `X` has no confounders, so the simple regression `Y ~ X` is the right model.

## Summary

| Model | Avg coefficient of X | Good model? |
|---|---|---|
| Y ~ X | **≈ −9.65** | ✅ unbiased for this data |
| Y ~ X + Z | **≈ −10.32** | ❌ collider bias |

Across runs the spread is tiny (standard deviation about 0.05), so you'll get essentially the same numbers.